# High-Quality Merged Data Visualization

**Purpose**: Generate publication-ready visualizations for merged reference+query data

**Input**: `reference_plus_query_merged_L2.h5ad`

**Output**: 6 independent PDFs (300 DPI) + summary report

## 1. Setup and Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc

print("✓ Libraries imported")
print(f"scanpy: {sc.__version__}")
print(f"numpy: {np.__version__}")

✓ Libraries imported
scanpy: 1.11.5
numpy: 2.2.5


In [2]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

# Input file
MERGED_H5AD = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/reference_plus_query_merged_L2.h5ad"

# Output directory
OUTPUT_DIR = "./visualizations_final"

# Visualization parameters
DPI = 300
FIGURE_FORMAT = 'pdf'
CONFIDENCE_THRESHOLD = 0.5

# Color schemes
DATA_SOURCE_COLORS = {
    'reference': '#3498db',
    'query': '#e74c3c'
}

CELLTYPE_COLORS = {
    'Plasma': '#e74c3c',
    'Naive_B': '#3498db',
    'Memory_B': '#2ecc71',
    'Atypical_Memory_B': '#f39c12',
    'GC_B': '#9b59b6',
    'Unknown': '#95a5a6'
}

print(f"Configuration loaded:")
print(f"  Input: {MERGED_H5AD}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  DPI: {DPI}")
print(f"  Format: {FIGURE_FORMAT}")

Configuration loaded:
  Input: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/reference_plus_query_merged_L2.h5ad
  Output: ./visualizations_final
  DPI: 300
  Format: pdf


In [3]:
# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Set scanpy parameters
sc.settings.set_figure_params(dpi=DPI, facecolor='white', format=FIGURE_FORMAT)
sc.settings.verbosity = 1

print(f"✓ Output directory created: {output_dir}")

✓ Output directory created: visualizations_final


## 2. Helper Functions

In [4]:
def cleanup_temp_columns(adata, col_names):
    """
    Remove temporary columns from adata.obs.
    """
    if isinstance(col_names, str):
        col_names = [col_names]
    
    for col in col_names:
        if col in adata.obs.columns:
            adata.obs.drop(columns=[col], inplace=True)

print("✓ Helper functions defined")

✓ Helper functions defined


## 3. Load and Validate Data

In [5]:
print("Loading merged data...")
adata = sc.read_h5ad(MERGED_H5AD)

print(f"\nDataset info:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  obs columns: {len(adata.obs.columns)}")
print(f"  obsm keys: {list(adata.obsm.keys())}")

Loading merged data...

Dataset info:
  Cells: 52,787
  Genes: 3,955
  obs columns: 51
  obsm keys: ['X_scANVI_L2', 'X_umap']


In [6]:
# Validate required columns
required_cols = ['data_source']
missing = [c for c in required_cols if c not in adata.obs.columns]

if missing:
    print(f"ERROR: Missing required columns: {missing}")
else:
    print(f"✓ All required columns present")

# Check UMAP
if 'X_umap' not in adata.obsm:
    print(f"ERROR: X_umap not found in obsm")
else:
    print(f"✓ X_umap found: {adata.obsm['X_umap'].shape}")

✓ All required columns present
✓ X_umap found: (52787, 2)


In [7]:
# Split reference and query
ref_mask = adata.obs['data_source'] == 'reference'
qry_mask = adata.obs['data_source'] == 'query'

ref_cells = adata[ref_mask]
qry_cells = adata[qry_mask]

print(f"\nData split:")
print(f"  Reference: {ref_cells.n_obs:,} cells")
print(f"  Query: {qry_cells.n_obs:,} cells")

if ref_cells.n_obs == 0:
    print("WARNING: No reference cells!")
if qry_cells.n_obs == 0:
    print("WARNING: No query cells!")


Data split:
  Reference: 11,570 cells
  Query: 41,217 cells


## 4. Figure 1: UMAP Overview (2×2)

In [8]:
print("\nGenerating Figure 1: UMAP Overview...")

fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# Panel 1: Data source
print("  Panel 1: Data source...")
sc.pl.umap(adata, color='data_source', ax=axes[0, 0], show=False,
           title='Data Source', s=8, frameon=True,
           palette=DATA_SOURCE_COLORS, legend_loc='right margin')

print("✓ Panel 1 complete")


Generating Figure 1: UMAP Overview...
  Panel 1: Data source...
✓ Panel 1 complete


In [9]:
# Panel 2: Reference L2
print("  Panel 2: Reference L2...")
temp_col_name = '_temp_ref_L2'

try:
    if 'Cell_Type_L2' in adata.obs.columns:
        temp_col = pd.Series('N/A', index=adata.obs_names, dtype='object')
        ref_mask = adata.obs['data_source'] == 'reference'
        
        if ref_mask.any():
            temp_col[ref_mask] = adata.obs.loc[ref_mask, 'Cell_Type_L2'].astype(str)
            temp_col = temp_col.astype('category')
            adata.obs[temp_col_name] = temp_col
            
            sc.pl.umap(adata, color=temp_col_name, ax=axes[0, 1], show=False,
                       title='Reference L2 Labels', s=8, frameon=True,
                       legend_loc='right margin', na_color='lightgray')
        else:
            axes[0, 1].text(0.5, 0.5, 'No reference cells',
                           ha='center', va='center', transform=axes[0, 1].transAxes,
                           fontsize=14, color='gray')
            axes[0, 1].set_title('Reference L2 Labels')
    else:
        axes[0, 1].text(0.5, 0.5, 'Reference L2 not available',
                       ha='center', va='center', transform=axes[0, 1].transAxes,
                       fontsize=14, color='gray')
        axes[0, 1].set_title('Reference L2 Labels')
except Exception as e:
    print(f"  WARNING: Panel 2 failed: {e}")
    axes[0, 1].text(0.5, 0.5, 'Error generating panel',
                   ha='center', va='center', transform=axes[0, 1].transAxes,
                   fontsize=14, color='red')
finally:
    cleanup_temp_columns(adata, temp_col_name)

print("✓ Panel 2 complete")

  Panel 2: Reference L2...
✓ Panel 2 complete


In [10]:
# Panel 3: Query L2 final
print("  Panel 3: Query L2 predictions...")
temp_col_name = '_temp_qry_L2'

try:
    if 'Cell_Type_L2_final' in adata.obs.columns:
        temp_col = pd.Series('N/A', index=adata.obs_names, dtype='object')
        qry_mask = adata.obs['data_source'] == 'query'
        
        if qry_mask.any():
            temp_col[qry_mask] = adata.obs.loc[qry_mask, 'Cell_Type_L2_final'].astype(str)
            temp_col = temp_col.astype('category')
            adata.obs[temp_col_name] = temp_col
            
            sc.pl.umap(adata, color=temp_col_name, ax=axes[1, 0], show=False,
                       title=f'Query L2 Predictions (conf ≥ {CONFIDENCE_THRESHOLD})', 
                       s=8, frameon=True, legend_loc='right margin',
                       palette=CELLTYPE_COLORS, na_color='lightgray')
        else:
            axes[1, 0].text(0.5, 0.5, 'No query data',
                           ha='center', va='center', transform=axes[1, 0].transAxes,
                           fontsize=14, color='gray')
            axes[1, 0].set_title('Query L2 Predictions')
    else:
        axes[1, 0].text(0.5, 0.5, 'Query L2 not available',
                       ha='center', va='center', transform=axes[1, 0].transAxes,
                       fontsize=14, color='gray')
        axes[1, 0].set_title('Query L2 Predictions')
except Exception as e:
    print(f"  WARNING: Panel 3 failed: {e}")
    axes[1, 0].text(0.5, 0.5, 'Error generating panel',
                   ha='center', va='center', transform=axes[1, 0].transAxes,
                   fontsize=14, color='red')
finally:
    cleanup_temp_columns(adata, temp_col_name)

print("✓ Panel 3 complete")

  Panel 3: Query L2 predictions...
✓ Panel 3 complete


In [11]:
# Panel 4: Query confidence
print("  Panel 4: Query mapping confidence...")
temp_col_name = '_temp_conf'

try:
    if 'mapping_confidence' in adata.obs.columns:
        temp_conf = np.full(adata.n_obs, np.nan, dtype=float)
        qry_mask = adata.obs['data_source'] == 'query'
        
        if qry_mask.any():
            temp_conf[qry_mask] = pd.to_numeric(
                adata.obs.loc[qry_mask, 'mapping_confidence'], 
                errors='coerce'
            ).values
            
            adata.obs[temp_col_name] = temp_conf
            
            sc.pl.umap(adata, color=temp_col_name, ax=axes[1, 1], show=False,
                       title='Query Mapping Confidence', s=8, frameon=True,
                       cmap='RdYlGn', vmin=0, vmax=1, na_color='lightgray')
        else:
            axes[1, 1].text(0.5, 0.5, 'No query data',
                           ha='center', va='center', transform=axes[1, 1].transAxes,
                           fontsize=14, color='gray')
            axes[1, 1].set_title('Query Mapping Confidence')
    else:
        axes[1, 1].text(0.5, 0.5, 'Confidence not available',
                       ha='center', va='center', transform=axes[1, 1].transAxes,
                       fontsize=14, color='gray')
        axes[1, 1].set_title('Query Mapping Confidence')
except Exception as e:
    print(f"  WARNING: Panel 4 failed: {e}")
    axes[1, 1].text(0.5, 0.5, 'Error generating panel',
                   ha='center', va='center', transform=axes[1, 1].transAxes,
                   fontsize=14, color='red')
finally:
    cleanup_temp_columns(adata, temp_col_name)

print("✓ Panel 4 complete")

  Panel 4: Query mapping confidence...
✓ Panel 4 complete


In [12]:
# Save Figure 1
plt.tight_layout()
fig_path = output_dir / f"01_umap_overview.{FIGURE_FORMAT}"

try:
    plt.savefig(fig_path, dpi=DPI, bbox_inches='tight')
    print(f"✓ Saved: {fig_path.name}")
except Exception as e:
    print(f"ERROR saving figure: {e}")
finally:
    plt.close()

print("\n✓ Figure 1 complete!")

✓ Saved: 01_umap_overview.pdf

✓ Figure 1 complete!


## 5. Figure 2: Reference vs Query Side-by-Side (1×2)

In [13]:
print("\nGenerating Figure 2: Side-by-Side Comparison...")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel 1: Reference only
print("  Panel 1: Reference cells...")
sc.pl.umap(ref_cells, ax=axes[0], show=False,
           title=f'Reference Only ({ref_cells.n_obs:,} cells)',
           color=DATA_SOURCE_COLORS['reference'], s=8, frameon=True)

# Panel 2: Query only
print("  Panel 2: Query cells...")
sc.pl.umap(qry_cells, ax=axes[1], show=False,
           title=f'Query Only ({qry_cells.n_obs:,} cells)',
           color=DATA_SOURCE_COLORS['query'], s=8, frameon=True)

plt.tight_layout()
fig_path = output_dir / f"02_reference_vs_query_sidebyside.{FIGURE_FORMAT}"

try:
    plt.savefig(fig_path, dpi=DPI, bbox_inches='tight')
    print(f"✓ Saved: {fig_path.name}")
except Exception as e:
    print(f"ERROR saving figure: {e}")
finally:
    plt.close()

print("\n✓ Figure 2 complete!")


Generating Figure 2: Side-by-Side Comparison...
  Panel 1: Reference cells...


KeyError: 'Could not find key #3498db in .var_names or .obs.columns.'

## 6. Figure 3: UMAP Space Validation (1×3)

In [ ]:
print("\nGenerating Figure 3: UMAP Space Validation...")

if ref_cells.n_obs > 0 and qry_cells.n_obs > 0:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    ref_umap = ref_cells.obsm['X_umap']
    qry_umap = qry_cells.obsm['X_umap']
    
    # Panel 1: X coordinate distribution
    print("  Panel 1: X coordinate distribution...")
    axes[0].hist(ref_umap[:, 0], bins=50, alpha=0.5, 
                label=f'Reference (n={ref_cells.n_obs:,})',
                color=DATA_SOURCE_COLORS['reference'], density=True)
    axes[0].hist(qry_umap[:, 0], bins=50, alpha=0.5,
                label=f'Query (n={qry_cells.n_obs:,})',
                color=DATA_SOURCE_COLORS['query'], density=True)
    axes[0].set_xlabel('UMAP X', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_title('UMAP X Coordinate Distribution', 
                     fontsize=14, fontweight='bold')
    axes[0].legend(frameon=True)
    axes[0].grid(alpha=0.3)
    
    # Panel 2: Y coordinate distribution
    print("  Panel 2: Y coordinate distribution...")
    axes[1].hist(ref_umap[:, 1], bins=50, alpha=0.5,
                label=f'Reference (n={ref_cells.n_obs:,})',
                color=DATA_SOURCE_COLORS['reference'], density=True)
    axes[1].hist(qry_umap[:, 1], bins=50, alpha=0.5,
                label=f'Query (n={qry_cells.n_obs:,})',
                color=DATA_SOURCE_COLORS['query'], density=True)
    axes[1].set_xlabel('UMAP Y', fontsize=12)
    axes[1].set_ylabel('Density', fontsize=12)
    axes[1].set_title('UMAP Y Coordinate Distribution',
                     fontsize=14, fontweight='bold')
    axes[1].legend(frameon=True)
    axes[1].grid(alpha=0.3)
    
    # Panel 3: 2D overlay
    print("  Panel 3: 2D overlay...")
    axes[2].scatter(ref_umap[:, 0], ref_umap[:, 1], s=5, alpha=0.3,
                   c=DATA_SOURCE_COLORS['reference'], 
                   label=f'Reference ({ref_cells.n_obs:,})')
    axes[2].scatter(qry_umap[:, 0], qry_umap[:, 1], s=5, alpha=0.3,
                   c=DATA_SOURCE_COLORS['query'],
                   label=f'Query ({qry_cells.n_obs:,})')
    axes[2].set_xlabel('UMAP X', fontsize=12)
    axes[2].set_ylabel('UMAP Y', fontsize=12)
    axes[2].set_title('UMAP Space Overlap', fontsize=14, fontweight='bold')
    axes[2].legend(frameon=True)
    axes[2].grid(alpha=0.3)
    
    # Add statistics (FIXED: use np.ptp instead of .ptp())
    ref_x_range = np.ptp(ref_umap[:, 0])
    ref_y_range = np.ptp(ref_umap[:, 1])
    qry_x_range = np.ptp(qry_umap[:, 0])
    qry_y_range = np.ptp(qry_umap[:, 1])
    
    stats_text = f"Range Comparison:\n"
    stats_text += f"Ref: X={ref_x_range:.1f}, Y={ref_y_range:.1f}\n"
    stats_text += f"Qry: X={qry_x_range:.1f}, Y={qry_y_range:.1f}\n"
    
    # Safe division
    if ref_x_range > 0 and ref_y_range > 0:
        stats_text += f"Ratio: X={qry_x_range/ref_x_range:.2f}, Y={qry_y_range/ref_y_range:.2f}"
    else:
        stats_text += f"Ratio: N/A (ref range is zero)"
    
    axes[2].text(0.05, 0.95, stats_text, transform=axes[2].transAxes,
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    fig_path = output_dir / f"03_umap_space_validation.{FIGURE_FORMAT}"
    
    try:
        plt.savefig(fig_path, dpi=DPI, bbox_inches='tight')
        print(f"✓ Saved: {fig_path.name}")
    except Exception as e:
        print(f"ERROR saving figure: {e}")
    finally:
        plt.close()
else:
    print("  Skipping: Need both reference and query cells")

print("\n✓ Figure 3 complete!")

## 7. Display Summary

In [ ]:
print("\n" + "=" * 70)
print("VISUALIZATION SUMMARY")
print("=" * 70)

print(f"\nDataset:")
print(f"  Total cells: {adata.n_obs:,}")
print(f"  Reference: {ref_cells.n_obs:,} ({ref_cells.n_obs/adata.n_obs*100:.1f}%)")
print(f"  Query: {qry_cells.n_obs:,} ({qry_cells.n_obs/adata.n_obs*100:.1f}%)")
print(f"  Genes: {adata.n_vars:,}")

if 'mapping_confidence' in adata.obs.columns:
    conf = pd.to_numeric(
        adata.obs.loc[qry_mask, 'mapping_confidence'],
        errors='coerce'
    )
    conf_valid = conf[conf.notna()]
    
    if len(conf_valid) > 0:
        print(f"\nMapping Quality:")
        print(f"  Confidence mean: {conf_valid.mean():.3f}")
        print(f"  Confidence median: {conf_valid.median():.3f}")
        high_conf = (conf_valid >= CONFIDENCE_THRESHOLD).sum()
        print(f"  High confidence (≥{CONFIDENCE_THRESHOLD}): {high_conf:,} ({high_conf/len(conf_valid)*100:.1f}%)")

print(f"\nGenerated Figures:")
for pdf in sorted(output_dir.glob("*.pdf")):
    file_size = pdf.stat().st_size / 1024
    print(f"  {pdf.name} ({file_size:.1f} KB)")

print(f"\n✓ Visualization complete!")
print(f"Output directory: {output_dir}")